# EXP20: 最优参数组合验证

等 EXP01-19 全部出结果后，汇总各维度冠军参数，组合测试。

1. 当前 baseline
2. 各维度独立冠军
3. 冠军组合 K-Fold
4. 冠军组合 vs baseline 对比

In [ ]:
import sys; sys.path.insert(0, '..')
from backtest import DataBundle, run_variant, run_kfold, C12_KWARGS
from backtest.stats import print_ranked
import config
import pandas as pd
import numpy as np

data = DataBundle.load_default()

LIVE_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": 0.35,
    "kc_only_threshold": 0.60,
    "spread_cost": 0.50,
}
print('Data loaded')

## Step 1: 从各 EXP 填入冠军参数

根据 EXP01-19 的结果，逐个填入每个维度的最优值。
只填入 K-Fold 验证通过、非参数悬崖的参数。

In [ ]:
# =========================================
# 从各 EXP 结果填入冠军参数
# 只修改有明确改善的维度，其余保持 baseline
# =========================================

# EXP01: KC max_hold
BEST_KC_MH = 15  # <-- 从 EXP01 填入

# EXP04: SL / Trail
BEST_SL = 4.5            # <-- 从 EXP04 填入
BEST_TRAIL_ACT = 0.8     # <-- 从 EXP04 填入
BEST_TRAIL_DIST = 0.25   # <-- 从 EXP04 填入

# EXP09: ADX
BEST_ADX = 18             # <-- 从 EXP09 填入

# EXP11: TP
BEST_TP = 5.0             # <-- 从 EXP11 填入

# EXP15: Adaptive thresholds
BEST_CHOPPY = 0.35        # <-- 从 EXP15 填入
BEST_KC_ONLY = 0.60       # <-- 从 EXP15 填入

In [ ]:
# Baseline
baseline = run_variant(data, "Baseline", **LIVE_KWARGS)

# Champion combo
original_mh = config.STRATEGIES['keltner']['max_hold_bars']
config.STRATEGIES['keltner']['max_hold_bars'] = BEST_KC_MH

CHAMP_KWARGS = {
    **C12_KWARGS,
    "intraday_adaptive": True,
    "choppy_threshold": BEST_CHOPPY,
    "kc_only_threshold": BEST_KC_ONLY,
    "spread_cost": 0.50,
    "sl_atr_mult": BEST_SL,
    "tp_atr_mult": BEST_TP,
    "trailing_activate_atr": BEST_TRAIL_ACT,
    "trailing_distance_atr": BEST_TRAIL_DIST,
    "keltner_adx_threshold": BEST_ADX,
}
champion = run_variant(data, "Champion_Combo", **CHAMP_KWARGS)
config.STRATEGIES['keltner']['max_hold_bars'] = original_mh

print(f"Baseline:  Sharpe={baseline['sharpe']:.2f}, PnL=${baseline['total_pnl']:.0f}, N={baseline['n']}")
print(f"Champion:  Sharpe={champion['sharpe']:.2f}, PnL=${champion['total_pnl']:.0f}, N={champion['n']}")
print(f"Delta:     Sharpe={champion['sharpe']-baseline['sharpe']:+.2f}, PnL=${champion['total_pnl']-baseline['total_pnl']:+.0f}")

In [ ]:
# K-Fold comparison
print("\n=== K-Fold: Baseline vs Champion Combo ===")
config.STRATEGIES['keltner']['max_hold_bars'] = BEST_KC_MH
champ_folds = run_kfold(data, CHAMP_KWARGS, label_prefix="Champ_")
config.STRATEGIES['keltner']['max_hold_bars'] = original_mh
base_folds = run_kfold(data, LIVE_KWARGS, label_prefix="Base_")

for b, c in zip(base_folds, champ_folds):
    print(f"{b['fold']}: Base={b['sharpe']:.2f}  Champ={c['sharpe']:.2f}  D={c['sharpe']-b['sharpe']:+.2f}")

wins = sum(1 for b, c in zip(base_folds, champ_folds) if c['sharpe'] > b['sharpe'])
print(f"\nChampion wins {wins}/{len(base_folds)} folds")